---
## Section 1 — Setup & Configuration

In [1]:
import os, time, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision.models import resnet18
from sklearn.metrics import roc_curve, roc_auc_score
warnings.filterwarnings('ignore')

# ── Hyperparameters ────────────────────────────────────────────────────────
N_SHADOW    = 64       # number of shadow models
EPOCHS      = 50       # epochs per shadow model
BATCH_SIZE  = 256
LR          = 0.1
SHADOW_FRAC = 0.5      # fraction of pub used per shadow
N_AUG       = 12       # TTA augmentations for score computation

# ── Paths ──────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = Path('./shadow_checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)
BASE       = Path('.')
PUB_PATH   = BASE / 'pub.pt'
PRIV_PATH  = BASE / 'priv.pt'
MODEL_PATH = BASE / 'model.pt'
OUTPUT_CSV = BASE / 'submission.csv'

# ── Dataset stats (pre-computed) ───────────────────────────────────────────
MEAN = [0.7406, 0.5331, 0.7059]
STD  = [0.1491, 0.1864, 0.1301]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.backends.cudnn.benchmark = True

Using device: cuda


---
## Section 2 — Dataset & Model Loading

In [2]:
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids, self.imgs, self.labels = [], [], []
        self.transform = transform
    def __getitem__(self, index):
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        return self.ids[index], img, self.labels[index]
    def __len__(self):
        return len(self.ids)

class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []
    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]

transform = T.Compose([
    T.Resize(32),
    T.Normalize(mean=MEAN, std=STD),
])

print('Loading datasets...')
pub_ds  = torch.load(PUB_PATH,  weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)
pub_ds.transform  = transform
priv_ds.transform = transform
priv_ds.membership = [-1] * len(priv_ds.ids)

print('Loading target model...')
model = resnet18(weights=None)
model.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
model.maxpool = nn.Identity()
model.fc      = nn.Linear(512, 9)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval().to(device)

pub_loader  = DataLoader(pub_ds,  batch_size=512, shuffle=False, num_workers=0, pin_memory=True)
priv_loader = DataLoader(priv_ds, batch_size=512, shuffle=False, num_workers=0, pin_memory=True)

N_PUB  = len(pub_ds)
N_PRIV = len(priv_ds)
print(f'pub_ds: {N_PUB} | priv_ds: {N_PRIV}')

def make_model():
    m = resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(512, 9)
    return m.to(device)

Loading datasets...
Loading target model...
pub_ds: 14000 | priv_ds: 14000


---
## Section 3 — Scoring Functions (12-TTA Log-Odds)

In [ ]:
def get_aug_fns(n_aug):
    
    all_fns = [
        lambda x: x,                                               # 1. identity
        lambda x: torch.flip(x, dims=[3]),                         # 2. h-flip
        lambda x: torch.flip(x, dims=[2]),                         # 3. v-flip
        lambda x: torch.roll(x, shifts=2,   dims=3),               # 4. roll right
        lambda x: torch.roll(x, shifts=-2,  dims=3),               # 5. roll left
        lambda x: torch.roll(x, shifts=2,   dims=2),               # 6. roll down
        lambda x: torch.roll(x, shifts=-2,  dims=2),               # 7. roll up
        lambda x: (x * 0.92).clamp(0, 1),                          # 8. darken
        lambda x: (x * 1.08).clamp(0, 1),                          # 9. brighten
        lambda x: (x + 0.008 * torch.randn_like(x)).clamp(0, 1),   # 10. noise
        lambda x: torch.flip(x, dims=[2, 3]),                      # 11. 180° rot
        lambda x: (x * 0.96 + 0.02),                               # 12. contrast squeeze
    ]
    return all_fns[:n_aug]


def compute_logit_scores(net, loader, n_aug=N_AUG):
    
    net.eval()
    all_ids, all_scores, all_labels, all_members = [], [], [], []
    aug_fns = get_aug_fns(n_aug)
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 4:
                ids, imgs, labels, membership = batch
                all_members.extend(membership.tolist())
            else:
                ids, imgs, labels = batch
            all_ids.extend(ids.tolist() if isinstance(ids, torch.Tensor) else ids)
            all_labels.extend(labels.tolist())
            imgs, labels = imgs.to(device), labels.to(device)
            aug_scores = []
            for aug_fn in aug_fns:
                probs  = torch.softmax(net(aug_fn(imgs)), dim=1)
                p_y    = probs[torch.arange(len(labels)), labels].clamp(1e-7, 1-1e-7)
                aug_scores.append(torch.log(p_y / (1 - p_y)))
            all_scores.extend(torch.stack(aug_scores, dim=1).mean(dim=1).cpu().tolist())
    result = {'ids': all_ids, 'scores': np.array(all_scores), 'labels': np.array(all_labels)}
    if all_members:
        result['members'] = np.array(all_members)
    return result


def tpr_at_fpr(scores, labels, fpr_thr=0.05):
    fpr, tpr, _ = roc_curve(labels, scores)
    return float(tpr[np.where(fpr <= fpr_thr)[0][-1]])


def rank_norm(x):
    return np.argsort(np.argsort(x)).astype(float) / (len(x) - 1)


def per_class_demean(scores, labels, medians=None):
    out, meds = scores.copy(), {}
    for c in range(9):
        mask = (labels == c)
        med  = medians[c] if medians is not None else (np.median(scores[mask]) if mask.sum() > 0 else 0.0)
        meds[c]    = med
        out[mask] -= med
    return out, meds


print('Score functions defined.')

Score functions defined.


---
## Section 4 — Shadow Model Training

Trains 64 ResNet-18 shadow models, each on a random 50% subset of `pub.pt`.  
**Fully cache-aware**: if `shadow_checkpoints/shadow_XXX.pt` exists, training is skipped.  
Also computes and caches 12-TTA log-odds scores as `scores_v2_XXX.npz`.

In [ ]:
def train_shadow_and_score(shadow_id):
    
    ckpt_path     = CHECKPOINT_DIR / f'shadow_{shadow_id:03d}.pt'
    scores_v2_path = CHECKPOINT_DIR / f'scores_v2_{shadow_id:03d}.npz'

    rng     = np.random.RandomState(seed=shadow_id * 1337 + 42)
    indices = rng.permutation(N_PUB)
    n_in    = int(N_PUB * SHADOW_FRAC)
    in_idx  = indices[:n_in]
    in_mask = np.zeros(N_PUB, dtype=bool)
    in_mask[in_idx] = True

    if scores_v2_path.exists():
        d = np.load(scores_v2_path)
        return d['in_mask'], d['pub_scores'], d['priv_scores']

    shadow = make_model()
    if ckpt_path.exists():
        print(f'  [Shadow {shadow_id:03d}] Loading cached weights, recomputing v2 scores...')
        shadow.load_state_dict(torch.load(ckpt_path, map_location=device))
    else:
        print(f'  [Shadow {shadow_id:03d}] Training from scratch...')
        shadow_loader = DataLoader(
            Subset(pub_ds, in_idx.tolist()),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
        )
        aug = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomCrop(32, padding=4, padding_mode='reflect'),
        ])
        optimizer = optim.SGD(shadow.parameters(), lr=LR, momentum=0.9,
                              weight_decay=5e-4, nesterov=True)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        criterion = nn.CrossEntropyLoss()
        shadow.train()
        for epoch in range(EPOCHS):
            for batch in shadow_loader:
                _, imgs, labels = batch[:3]
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                criterion(shadow(aug(imgs)), labels).backward()
                optimizer.step()
            scheduler.step()
            if (epoch + 1) % 10 == 0:
                print(f'    Epoch {epoch+1}/{EPOCHS}')
        shadow.eval()
        torch.save(shadow.state_dict(), ckpt_path)

    shadow.eval()
    pub_scores  = compute_logit_scores(shadow, pub_loader,  n_aug=N_AUG)['scores']
    priv_scores = compute_logit_scores(shadow, priv_loader, n_aug=N_AUG)['scores']
    np.savez(scores_v2_path, in_mask=in_mask, pub_scores=pub_scores, priv_scores=priv_scores)
    del shadow
    torch.cuda.empty_cache()
    return in_mask, pub_scores, priv_scores


t0 = time.time()
print(f'[Step 1] Processing {N_SHADOW} shadow models...')

pub_in_mat  = np.full((N_PUB,  N_SHADOW), np.nan)
pub_out_mat = np.full((N_PUB,  N_SHADOW), np.nan)
priv_mat    = np.zeros((N_PRIV, N_SHADOW))

for i in range(N_SHADOW):
    in_mask, pub_s, priv_s = train_shadow_and_score(i)
    pub_in_mat[in_mask,   i] = pub_s[in_mask]
    pub_out_mat[~in_mask, i] = pub_s[~in_mask]
    priv_mat[:, i]           = priv_s
    if (i + 1) % 16 == 0:
        print(f'  {i+1}/{N_SHADOW} done | {(time.time()-t0)/60:.1f} min elapsed')

print(f'All {N_SHADOW} shadows ready in {(time.time()-t0)/60:.1f} min')

[Step 1] Processing 64 shadow models...
  16/64 done | 0.0 min elapsed
  32/64 done | 0.0 min elapsed
  48/64 done | 0.0 min elapsed
  64/64 done | 0.0 min elapsed
All 64 shadows ready in 0.0 min


---
## Section 5 — Target Model Scores & Attack Signal Construction



In [ ]:
from scipy.stats import norm as scipy_norm

print('[Step 2] Computing target model scores (12 TTA)...')
r_pub  = compute_logit_scores(model, pub_loader)
r_priv = compute_logit_scores(model, priv_loader)

pub_ts   = r_pub['scores']
priv_ts  = r_priv['scores']
pub_mem  = np.array(r_pub['members'])
pub_ids  = r_pub['ids']
priv_ids = r_priv['ids']

pub_labels_arr  = np.array([pub_ds[i][2]  for i in range(N_PUB)])
priv_labels_arr = np.array([priv_ds[i][2] for i in range(N_PRIV)])

print('[Step 3] Computing all RMIA variants...')

def winsorized_mean(arr, pct=0.10):
    lo = np.nanpercentile(arr, pct * 100)
    hi = np.nanpercentile(arr, (1 - pct) * 100)
    return np.nanmean(np.clip(arr, lo, hi))

pub_out_mean  = np.nanmean(pub_out_mat, axis=1)
pub_out_med   = np.nanmedian(pub_out_mat, axis=1)
pub_out_wmean = np.array([winsorized_mean(pub_out_mat[j, ~np.isnan(pub_out_mat[j])]) for j in range(N_PUB)])
pub_out_p25   = np.nanpercentile(pub_out_mat, 25, axis=1)
pub_out_p75   = np.nanpercentile(pub_out_mat, 75, axis=1)

rmia_mean  = pub_ts - pub_out_mean
rmia_med   = pub_ts - pub_out_med
rmia_wmean = pub_ts - pub_out_wmean
rmia_p25   = pub_ts - pub_out_p25
rmia_p75   = pub_ts - pub_out_p75

lira_proper = np.zeros(N_PUB)
for j in range(N_PUB):
    in_s  = pub_in_mat[j,  ~np.isnan(pub_in_mat[j])]
    out_s = pub_out_mat[j, ~np.isnan(pub_out_mat[j])]
    if len(in_s) < 3 or len(out_s) < 3:
        continue
    mu_in,  si_in  = in_s.mean(),  max(in_s.std(),  1e-4)
    mu_out, si_out = out_s.mean(), max(out_s.std(), 1e-4)
    ts = pub_ts[j]
    lira_proper[j] = (scipy_norm.logpdf(ts, mu_in,  si_in) -
                      scipy_norm.logpdf(ts, mu_out, si_out))

rmia_mean_dm,  med_mean  = per_class_demean(rmia_mean,   pub_labels_arr)
rmia_med_dm,   med_med   = per_class_demean(rmia_med,    pub_labels_arr)
rmia_wmean_dm, med_wmean = per_class_demean(rmia_wmean,  pub_labels_arr)
lira_dm,       med_lira  = per_class_demean(lira_proper, pub_labels_arr)
rmia_p25_dm,   med_p25   = per_class_demean(rmia_p25,    pub_labels_arr)
rmia_p75_dm,   med_p75   = per_class_demean(rmia_p75,    pub_labels_arr)

print('\n  Pub signal summary:')
for name, sc in [('rmia_mean',    rmia_mean),  ('rmia_med',     rmia_med),
                 ('rmia_wmean',   rmia_wmean), ('rmia_p25',     rmia_p25),
                 ('rmia_p75',     rmia_p75),   ('lira_proper',  lira_proper),
                 ('rmia_mean_dm', rmia_mean_dm),('rmia_med_dm',  rmia_med_dm),
                 ('rmia_wmean_dm',rmia_wmean_dm),('lira_dm',     lira_dm),
                 ('rmia_p25_dm',  rmia_p25_dm)]:
    auc = roc_auc_score(pub_mem, sc)
    tpr = tpr_at_fpr(sc, pub_mem)
    print(f'    {name:<18} AUC={auc:.4f}  TPR@5%FPR={tpr:.4f}')

[Step 2] Computing target model scores (12 TTA)...
[Step 3] Computing all RMIA variants...

  Pub signal summary:
    rmia_mean          AUC=0.5095  TPR@5%FPR=0.0596
    rmia_med           AUC=0.5100  TPR@5%FPR=0.0600
    rmia_wmean         AUC=0.5096  TPR@5%FPR=0.0591
    rmia_p25           AUC=0.5100  TPR@5%FPR=0.0606
    rmia_p75           AUC=0.5093  TPR@5%FPR=0.0573
    lira_proper        AUC=0.5070  TPR@5%FPR=0.0526
    rmia_mean_dm       AUC=0.5136  TPR@5%FPR=0.0613
    rmia_med_dm        AUC=0.5143  TPR@5%FPR=0.0607
    rmia_wmean_dm      AUC=0.5137  TPR@5%FPR=0.0613
    lira_dm            AUC=0.5075  TPR@5%FPR=0.0516
    rmia_p25_dm        AUC=0.5139  TPR@5%FPR=0.0610


### Fixed blend weights
Found via exhaustive GPU grid search (10M combinations) on pub dataset.
Hardcoded here to avoid re-running the search and ensure reproducibility.

In [ ]:
# ── Hardcoded blend weights from GPU grid search ──────────────────────────
# rmia_med_dm  w=0.833
# rmia_p25_dm  w=0.167
W_MED_DM = 0.833
W_P25_DM = 0.167

pub_blend = W_MED_DM * rank_norm(rmia_med_dm) + W_P25_DM * rank_norm(rmia_p25_dm)
tpr_blend = tpr_at_fpr(pub_blend, pub_mem)
auc_blend = roc_auc_score(pub_mem, pub_blend)
print(f'  [LOCAL EVAL] Final blend  AUC={auc_blend:.4f}  TPR@5%FPR={tpr_blend:.4f}')
print(f'  (Expected: AUC=0.5143  TPR@5%FPR=0.0629)')

print('\n[Step 4] Building private scores...')
priv_out_med = np.nanmedian(priv_mat, axis=1)
priv_out_p25 = np.nanpercentile(priv_mat, 25, axis=1)

priv_rmia_med = priv_ts - priv_out_med
priv_rmia_p25 = priv_ts - priv_out_p25

priv_med_dm, _ = per_class_demean(priv_rmia_med, priv_labels_arr, medians=med_med)
priv_p25_dm, _ = per_class_demean(priv_rmia_p25, priv_labels_arr, medians=med_p25)

priv_blend = W_MED_DM * rank_norm(priv_med_dm) + W_P25_DM * rank_norm(priv_p25_dm)
priv_final = rank_norm(priv_blend)

print(f'  Score range: [{priv_final.min():.4f}, {priv_final.max():.4f}]')
print(f'  Score mean:  {priv_final.mean():.4f}  std: {priv_final.std():.4f}')

  [LOCAL EVAL] Final blend  AUC=0.5142  TPR@5%FPR=0.0629
  (Expected: AUC=0.5143  TPR@5%FPR=0.0629)

[Step 4] Building private scores...
  Score range: [0.0000, 1.0000]
  Score mean:  0.5000  std: 0.2887


---
## Section 6 — Generate & Save Submission

In [9]:
# ── Sanity checks ─────────────────────────────────────────────────────────
assert len(priv_final) == N_PRIV,               'Wrong number of scores'
assert np.all(priv_final >= 0),                  'Scores below 0'
assert np.all(priv_final <= 1),                  'Scores above 1'
assert not np.any(np.isnan(priv_final)),         'NaN scores found'

# ── Save submission ───────────────────────────────────────────────────────
submission_df = pd.DataFrame({
    'id':    [str(i) for i in priv_ids],
    'score': priv_final.astype(float)
})
submission_df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved {len(submission_df)} rows → {OUTPUT_CSV}  ✓')
print(f'Score range: [{priv_final.min():.4f}, {priv_final.max():.4f}]')
print(f'Score mean:  {priv_final.mean():.4f}  std: {priv_final.std():.4f}')
print(f'Total runtime: {(time.time()-t0)/60:.1f} min')

Saved 14000 rows → submission.csv  ✓
Score range: [0.0000, 1.0000]
Score mean:  0.5000  std: 0.2887
Total runtime: 3.4 min


---
## Section 7 — Submit to Leaderboard

In [ ]:
BASE_URL = ''
API_KEY  = ''   
TASK_ID  = ''

def submit_to_leaderboard():
    if not OUTPUT_CSV.exists():
        print(f'Error: {OUTPUT_CSV.name} not found.'); return
    print(f'Submitting {OUTPUT_CSV.name}...')
    try:
        with open(OUTPUT_CSV, 'rb') as f:
            resp = requests.post(
                f'{BASE_URL}/submit/{TASK_ID}',
                headers={'X-API-Key': API_KEY},
                files={'file': (OUTPUT_CSV.name, f, 'application/csv')},
                timeout=(10, 600),
            )
        try:    print('SUCCESS:', resp.json())
        except: print(resp.text)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Submission failed: {e}')

submit_to_leaderboard()